## Overview

This notebook integrates the completed Demographic, Behavioural Evidence, and Opportunity components into a final mobility representation. For each constrained representative population profile, it constructs trip-purpose probabilities and their corresponding temporal and travel-mode distributions. POI destination weights are then incorporated using the general Opportunity Layer and the industry-based refinement component. The integrated workflow is first tested on selected representative profiles before being applied to the full representative population space.

### 1. Import Libraries

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

from copy import deepcopy

In [2]:
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)

### 2. Set Up Project Paths

In [3]:
project_root = Path.cwd().parents[1]

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

### 3. Import Main Inputs

In [4]:
representative_population_path = (
    project_root
    / "output"
    / "demographic_profiles"
    / "constrained_representative_population_space.parquet"
)

behavioural_evidence_path = (
    project_root
    / "output"
    / "behavioural_evidence"
    / "nts_behavioural_evidence_layer.json"
)

opportunity_layer_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "opportunity_layer.json"
)

industry_opportunity_component_path = (
    project_root
    / "output"
    / "opportunity_layer"
    / "industry_to_poi_mapping.json"
)

In [5]:
representative_population = pd.read_parquet(
    representative_population_path
)

with open(
    behavioural_evidence_path,
    "r",
    encoding="utf-8",
) as file:
    assembled_behavioural_evidence = json.load(file)

with open(
    opportunity_layer_path,
    "r",
    encoding="utf-8",
) as file:
    opportunity_layer = json.load(file)

with open(
    industry_opportunity_component_path,
    "r",
    encoding="utf-8",
) as file:
    industry_component = json.load(file)

In [6]:
print(
    "Representative population shape:",
    representative_population.shape,
)

print(
    "Behavioural evidence purposes:",
    len(assembled_behavioural_evidence),
)

print(
    "Opportunity Layer entries:",
    len(opportunity_layer),
)

print(
    "Industry component entries:",
    len(industry_component),
)

Representative population shape: (2267232, 14)
Behavioural evidence purposes: 14
Opportunity Layer entries: 14
Industry component entries: 19


### 4. Import Main Functions

These functions integrate the Demographic Layer and Behavioural Evidence Layer and apply the relevant NTS constraints.

In [7]:
from src.demographic_behaviour_layers_integration import (
    construct_constrained_behaviour_profile,
    DEMOGRAPHIC_ATTRIBUTES
)

In [8]:
from src.complete_framework import (
    construct_complete_framework,
    display_complete_framework
)

### 5. Layer Integration: Demographic, Behavioural Evidence, Opportunity(Log Transformation, Industry Component)

In [10]:
import random

profile = representative_population.iloc[600000]

(
    complete_profile,
    framework_dataframes,
) = construct_complete_framework(
    representative_population_profile=profile,
    assembled_behavioural_evidence=(
        assembled_behavioural_evidence
    ),
    opportunity_layer=opportunity_layer,
    industry_component=industry_component,
)